## 5. Algoritmo de Grover
Busca un elemento marcado entre N = 2ⁿ con ~√N pasos (en vez de ~N).
Cada iteración = **oráculo** (marca la respuesta) + **difusor** (amplifica su probabilidad).

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()

def correr(qc, shots=1024):
    """Ejecuta un circuito en el simulador y devuelve los conteos."""
    return sim.run(transpile(qc, sim), shots=shots).result().get_counts()

In [ ]:
def grover(objetivo, iteraciones=None):
    n = len(objetivo)
    if iteraciones is None:
        iteraciones = int(np.floor(np.pi / 4 * np.sqrt(2**n)))

    def mcz(qc):  # Z multi-controlada sobre todos los qubits
        qc.h(n - 1); qc.mcx(list(range(n - 1)), n - 1); qc.h(n - 1)

    qc = QuantumCircuit(n, n)
    qc.h(range(n))
    for _ in range(iteraciones):
        # oráculo: invierte la fase del estado objetivo
        for i, bit in enumerate(reversed(objetivo)):
            if bit == "0": qc.x(i)
        mcz(qc)
        for i, bit in enumerate(reversed(objetivo)):
            if bit == "0": qc.x(i)
        # difusor: refleja alrededor del promedio
        qc.h(range(n)); qc.x(range(n))
        mcz(qc)
        qc.x(range(n)); qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

objetivo = "101"
conteos = correr(grover(objetivo))
print("Objetivo:", objetivo, "| Más frecuente:", max(conteos, key=conteos.get))
plot_histogram(conteos)